# Практична робота №3: перенавчання, складність моделі та регуляризація

**Варіант:** 7 — Bike Sharing Dataset (`cnt`), той самий, що в ЛР1 і ПР2.

Раніше прийняті рішення (не повторюються детально): хронологічний поділ 80/20, `TimeSeriesSplit(5)` на train, preprocessing у `Pipeline`, основна метрика RMSE. Нове питання: **чи відтворює модель закономірність, чи лише пристосовується до навчальних даних?**

In [1]:
from math import comb
from pathlib import Path
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.bike_workflow import (
    CATEGORICAL_FEATURES, NUMERIC_FEATURES, chronological_split, load_bike_data, split_features_target,
)

WORK_DIR = Path.cwd().parent
FIGURES = WORK_DIR / "figures"
RESULTS = WORK_DIR / "results"
FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)

## 1. Гіпотези (сформульовані до запуску)

| № | Гіпотеза | Що обчислюємо | Підтвердження | Спростування |
|---|---|---|---|---|
| H1 | Зі зростанням степеня полінома `d` train RMSE не зростає | train RMSE для кожного `d`, крива валідації | train RMSE монотонно спадає | train RMSE зростає |
| H2 | Після певного `d` CV RMSE і розрив узагальнення `G = CV − train` зростають (перенавчання) | CV RMSE, `G`, std | є мінімум CV, далі CV і `G` ростуть | CV спадає до найбільшого `d` |
| H3 | Для перенавченого простору помірна регуляризація зменшить CV RMSE, `G` і норму коефіцієнтів | Ridge/Lasso/ElasticNet на однаковому просторі | CV RMSE і `G` менші за OLS принаймні на 20% | зменшення немає |
| H4 | Надмірна регуляризація спричинить недонавчання | train і CV RMSE від `alpha` | обидві похибки ростуть і зближуються | — |
| H5 | Регуляризований складний поліном **не** дасть суттєвої переваги над простою моделлю | порівняльна таблиця + правило 1-SE | різниця з найкращою простою моделлю менша за 1 SE | регуляризований поліном кращий більш ніж на 1 SE |

«Покращення» = менший середній CV RMSE; додатково враховуються розрив узагальнення, std між fold-ами та простота моделі.

## 2. Експериментальний протокол

| Елемент | Значення |
|---|---|
| Train / test | перші 13 903 год / останні 3 476 год (як у ЛР1, ПР2); test закритий до кроку 11 |
| CV | `TimeSeriesSplit(n_splits=5)`, однакові fold-и для всіх конфігурацій |
| Метрика | RMSE (основна); MAE, R² — лише для фінального test |
| Pipeline | `ColumnTransformer`: числові → `PolynomialFeatures(d)` → `StandardScaler`; категоріальні → `OneHotEncoder` → модель |
| Шкала складності | `d ∈ {1, 2, 3, 4, 5, 6, 8}` для 5 числових ознак |
| Регуляризація | простір `d = 5`; Ridge `alpha ∈ logspace(−2, 4, 13)`; Lasso `alpha ∈ logspace(−1.5, 1.5, 7)`; ElasticNet ті самі `alpha` × `l1_ratio ∈ {0.2, 0.5, 0.8}` |
| Бюджет | 7 + 1 (OLS d=5) + 13 + 7 + 21 = 49 конфігурацій основного пошуку + перевірка стійкості на 5 схемах CV |
| Правило вибору | мінімальний середній CV RMSE; серед моделей у межах `RMSE_min + SE_min` обирається найпростіша (one-standard-error rule) |

**Чому поліном лише по числових ознаках.** Після one-hot маємо 55 бінарних категоріальних ознак; поліном 2-го степеня по всіх 60 ознаках дав би $\binom{62}{2}-1 = 1890$ стовпців, а 3-го — 41 663, причому більшість — добутки взаємовиключних індикаторів. Методичка прямо радить не робити поліноміальні комбінації всіх one-hot ознак, тому категоріальна гілка залишається лінійною.

In [2]:
df = load_bike_data()
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = chronological_split(X, y)
cv = TimeSeriesSplit(n_splits=5)
DEGREES = [1, 2, 3, 4, 5, 6, 8]
N_CATEGORICAL = 55  # кількість one-hot стовпців (перевірено нижче)


def make_pipeline(degree, estimator):
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([("poly", PolynomialFeatures(degree=degree, include_bias=False)),
                              ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
    ])
    return Pipeline([("preprocessor", preprocessor), ("model", estimator)])


p = len(NUMERIC_FEATURES)
check = make_pipeline(3, LinearRegression()).named_steps["preprocessor"].fit(X_train)
n_poly_3 = check.named_transformers_["numeric"].named_steps["poly"].n_output_features_
n_cat = len(check.named_transformers_["categorical"].get_feature_names_out())
print(f"p = {p}; формула для d=3: C(8,3) − 1 = {comb(p + 3, 3) - 1}; PolynomialFeatures дав {n_poly_3}; one-hot стовпців: {n_cat}")

p = 5; формула для d=3: C(8,3) − 1 = 55; PolynomialFeatures дав 55; one-hot стовпців: 55


## 3–4. Шкала складності та діагностика недонавчання / перенавчання

In [3]:
rows, fold_rows = [], []
for degree in DEGREES:
    start = time.perf_counter()
    scores = cross_validate(make_pipeline(degree, LinearRegression()), X_train, y_train, cv=cv,
                            scoring="neg_root_mean_squared_error", return_train_score=True)
    elapsed = time.perf_counter() - start
    train_rmse, cv_rmse = -scores["train_score"], -scores["test_score"]
    n_poly = comb(p + degree, degree) - 1
    rows.append({"d": degree, "Поліном. ознак": n_poly, "Усього ознак": n_poly + N_CATEGORICAL,
                 "Train RMSE": train_rmse.mean(), "CV RMSE": cv_rmse.mean(), "CV std": cv_rmse.std(ddof=1),
                 "CV SE": cv_rmse.std(ddof=1) / np.sqrt(len(cv_rmse)), "G = CV − train": cv_rmse.mean() - train_rmse.mean(),
                 "Час CV, с": elapsed})
    for i, value in enumerate(cv_rmse, 1):
        fold_rows.append({"d": degree, "fold": i, "CV RMSE": value, "Train RMSE": train_rmse[i - 1]})
complexity = pd.DataFrame(rows)
complexity_folds = pd.DataFrame(fold_rows)
complexity.to_csv(RESULTS / "complexity_results.csv", index=False)
complexity_folds.to_csv(RESULTS / "complexity_folds.csv", index=False)
display(complexity.style.format({"Train RMSE": "{:.2f}", "CV RMSE": "{:,.2f}", "CV std": "{:,.2f}", "CV SE": "{:,.2f}",
                                 "G = CV − train": "{:,.2f}", "Час CV, с": "{:.1f}"}))
display(complexity_folds.pivot(index="d", columns="fold", values="CV RMSE").style.format("{:,.1f}"))

,d,Поліном. ознак,Усього ознак,Train RMSE,CV RMSE,CV std,CV SE,G = CV − train,"Час CV, с"
0,1,5,60,69.14,108.76,21.64,9.68,39.62,0.2
1,2,20,75,67.43,108.90,24.11,10.78,41.46,0.3
2,3,55,110,65.79,109.52,22.62,10.12,43.72,0.7
3,4,125,180,65.15,158.97,75.00,33.54,93.81,0.8
4,5,251,306,64.54,260.16,203.01,90.79,195.63,1.2
5,6,461,516,63.62,"2,337.20","2,355.38","1,053.36","2,273.58",2.0
6,8,1286,1341,61.06,"205,895.28","269,036.07","120,316.59","205,834.21",8.5


fold,1,2,3,4,5
d,,,,,
1,117.5,91.0,82.8,116.5,136.1
2,105.7,94.3,84.0,113.3,147.1
3,115.2,91.0,84.6,115.2,141.5
4,268.4,96.3,85.7,151.4,193.1
5,604.8,141.0,85.7,242.1,227.2
6,"3,677.2","1,312.6",89.6,"5,796.0",810.5
8,"499,489.5","11,346.0","4,183.2","501,683.9","12,773.7"


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.plot(complexity["d"], complexity["Train RMSE"], marker="o", label="Train RMSE", color="#3f7f93")
ax.errorbar(complexity["d"], complexity["CV RMSE"], yerr=complexity["CV std"], marker="o", capsize=4,
            label="CV RMSE ± std", color="#c46a5a")
ax.set_yscale("log")
ax.axvspan(0.7, 3.3, color="#6a8f3f", alpha=0.08, label="прийнятна складність")
ax.axvspan(3.7, 8.3, color="#c46a5a", alpha=0.08, label="перенавчання")
ax.set(title="Крива валідації (логарифмічна шкала)", xlabel="Степінь полінома d", ylabel="RMSE", xticks=DEGREES)
ax.legend(fontsize=8)
ax = axes[1]
zoom = complexity[complexity["d"] <= 4]
ax.plot(zoom["d"], zoom["Train RMSE"], marker="o", label="Train RMSE", color="#3f7f93")
ax.errorbar(zoom["d"], zoom["CV RMSE"], yerr=zoom["CV SE"], marker="o", capsize=4, label="CV RMSE ± SE", color="#c46a5a")
ax.set(title="Збільшено: d = 1…4 (лінійна шкала)", xlabel="Степінь полінома d", ylabel="RMSE", xticks=[1, 2, 3, 4])
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "validation_curve.png", dpi=160, bbox_inches="tight")
plt.show()

**Діагностика:**

- **H1 підтверджена:** train RMSE монотонно спадає від 69,1 (d=1) до 61,1 (d=8) — складніше сімейство містить простіші як окремі випадки.
- **Недонавчання** видно вже при d=1: train RMSE ≈ 69 і CV RMSE ≈ 109 обидва великі (ЛР1: HistGradientBoosting досягав CV ≈ 77). Але причина не в степені полінома: головна нелінійність — у взаємодії `hr × workingday`, а не в кривизні погодних ознак, тому d = 2–3 майже нічого не дають (108,9 і 109,5).
- **Найкраще узагальнення:** d = 1–3, CV RMSE 108,8–109,5; відмінності менші за SE ≈ 9–10.
- **Перенавчання:** з d = 4 CV RMSE зростає (159), при d = 6 — 2 337, при d = 8 — понад 200 000, тоді як train продовжує падати. **H2 підтверджена.**
- Механізм видно з fold-ів: катастрофічні похибки на fold 1 і fold 4, де validation-період (весна–літо) має температуру й вологість, не бачені в train цього fold-а. Поліном високого степеня **екстраполює** за межі навчального діапазону й «вибухає». Fold 3 (осінь–зима, діапазони вже покриті) стабільний для всіх `d`. Отже, перенавчання тут проявляється не як запам'ятовування шуму, а як нестабільна поведінка поза областю навчальних даних — саме те, що трапляється в реальному прогнозі майбутнього.

## 5. Криві навчання

Стандартний `learning_curve` з `TimeSeriesSplit` обмежує розмір навчальної підвибірки найменшим fold-ом (2 318 год) і бере **перші** години, тобто лише зиму–весну 2011 року. Тому крива побудована вручну, але за тим самим принципом: для трьох останніх часових fold-ів (validation-періоди 3, 4, 5 із `TimeSeriesSplit(5)`) модель навчається на `n` **найближчих у часі попередніх** годинах, `n ∈ {500, 1000, 2000, 4000, 6952}`; метрики усереднюються по трьох fold-ах.

In [5]:
SIZES = [500, 1000, 2000, 4000, 6952]
splits = list(cv.split(X_train))[2:]
lc_rows = []
for degree in [1, 5]:
    for n in SIZES:
        tr_scores, va_scores = [], []
        for tr, va in splits:
            idx = tr[-n:]
            model = make_pipeline(degree, LinearRegression()).fit(X_train.iloc[idx], y_train.iloc[idx])
            tr_scores.append(root_mean_squared_error(y_train.iloc[idx], model.predict(X_train.iloc[idx])))
            va_scores.append(root_mean_squared_error(y_train.iloc[va], model.predict(X_train.iloc[va])))
        lc_rows.append({"d": degree, "n_train": n, "Train RMSE": np.mean(tr_scores), "CV RMSE (mean)": np.mean(va_scores),
                        "CV RMSE (median)": np.median(va_scores)})
learning = pd.DataFrame(lc_rows)
learning.to_csv(RESULTS / "learning_curves.csv", index=False)
display(learning.style.format({"Train RMSE": "{:.1f}", "CV RMSE (mean)": "{:,.1f}", "CV RMSE (median)": "{:,.1f}"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, degree, title in [(axes[0], 1, "d = 1 (недостатня складність)"), (axes[1], 5, "d = 5 (перенавчання)")]:
    part = learning[learning["d"] == degree]
    ax.plot(part["n_train"], part["Train RMSE"], marker="o", label="Train RMSE", color="#3f7f93")
    ax.plot(part["n_train"], part["CV RMSE (mean)"], marker="o", label="CV RMSE (середнє 3 fold-ів)", color="#c46a5a")
    ax.plot(part["n_train"], part["CV RMSE (median)"], marker="s", ls="--", label="CV RMSE (медіана)", color="#8a6aa8")
    ax.set(title=f"Крива навчання: {title}", xlabel="Навчальних годин перед validation-періодом", ylabel="RMSE",
           xscale="log", xticks=SIZES)
    ax.set_xticklabels(SIZES)
    if degree == 5:
        ax.set_yscale("log")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "learning_curves.png", dpi=160, bbox_inches="tight")
plt.show()

,d,n_train,Train RMSE,CV RMSE (mean),CV RMSE (median)
0,1,500,80.2,111.8,115.3
1,1,1000,79.3,112.7,122.3
2,1,2000,80.5,110.6,120.3
3,1,4000,81.0,119.9,111.5
4,1,6952,79.7,109.5,114.2
5,5,500,62.5,"27,879.6","6,392.6"
6,5,1000,69.6,"16,712.4","5,852.9"
7,5,2000,74.7,"2,819.3","1,219.0"
8,5,4000,76.4,820.6,678.9
9,5,6952,75.3,202.1,243.1


- **d = 1:** криві плоскі — train RMSE ≈ 80, CV RMSE ≈ 110–120 незалежно від того, 500 чи 6 952 години історії. Висока похибка на обох кривих і відсутність прогресу — класичне **недонавчання (високе зміщення)**: більше даних не допоможе, потрібне інше представлення ознак (взаємодії `hr × workingday`) або нелінійна модель.
- **d = 5:** CV RMSE падає на два порядки — з ≈ 27 900 (500 год) до ≈ 202 (6 952 год), бо довша історія покриває ширший діапазон погоди й зменшує екстраполяцію. Розрив з train скорочується, але навіть на найбільшій підвибірці CV RMSE удвічі гірший, ніж у d=1, а train RMSE (≈ 75) нижчий. Це **висока дисперсія**: додаткові дані допомагають, але за доступного обсягу складність треба обмежувати. Шкала для d=5 логарифмічна, бо похибки відрізняються на порядки.

## 6–7. Регуляризація на перенавченому просторі d = 5 і вибір `alpha`

Простір d = 5 (251 поліноміальна + 55 one-hot ознак) обрано як явно перенавчений (CV RMSE 260 проти 109 у d=1), але ще не настільки нестабільний, як d ≥ 6. Структура pipeline, fold-и і метрика однакові для OLS, Ridge, Lasso та ElasticNet. Для Lasso/ElasticNet `max_iter=20 000`, `tol=1e-3`, `selection="random"`; попередження про незбіжність перехоплюються й підраховуються.

In [6]:
REG_DEGREE = 5
RIDGE_ALPHAS = np.logspace(-2, 4, 13)
L1_ALPHAS = np.logspace(-1.5, 1.5, 7)
L1_RATIOS = [0.2, 0.5, 0.8]


def make_estimator(name, alpha=None, l1_ratio=None):
    if name == "OLS":
        return LinearRegression()
    if name == "Ridge":
        return Ridge(alpha=alpha)
    if name == "Lasso":
        return Lasso(alpha=alpha, max_iter=20_000, tol=1e-3, selection="random", random_state=42)
    return ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20_000, tol=1e-3, selection="random", random_state=42)


def evaluate(name, alpha=None, l1_ratio=None, degree=REG_DEGREE, cv_scheme=cv):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        scores = cross_validate(make_pipeline(degree, make_estimator(name, alpha, l1_ratio)), X_train, y_train,
                                cv=cv_scheme, scoring="neg_root_mean_squared_error", return_train_score=True, n_jobs=2)
    train_rmse, cv_rmse = -scores["train_score"], -scores["test_score"]
    return {"Модель": name, "d": degree, "alpha": alpha, "l1_ratio": l1_ratio, "Train RMSE": train_rmse.mean(),
            "CV RMSE": cv_rmse.mean(), "CV std": cv_rmse.std(ddof=1), "CV SE": cv_rmse.std(ddof=1) / np.sqrt(len(cv_rmse)),
            "G": cv_rmse.mean() - train_rmse.mean(),
            "ConvergenceWarning": sum(issubclass(w.category, ConvergenceWarning) for w in caught)}


start = time.perf_counter()
reg_rows = [evaluate("OLS")]
reg_rows += [evaluate("Ridge", a) for a in RIDGE_ALPHAS]
reg_rows += [evaluate("Lasso", a) for a in L1_ALPHAS]
reg_rows += [evaluate("ElasticNet", a, r) for r in L1_RATIOS for a in L1_ALPHAS]
reg_search = pd.DataFrame(reg_rows)
reg_search.to_csv(RESULTS / "regularization_search.csv", index=False)
print(f"Оцінено конфігурацій: {len(reg_search)}; час: {time.perf_counter() - start:.0f} с; "
      f"попереджень про незбіжність: {reg_search['ConvergenceWarning'].sum()}")

best_reg = reg_search.loc[reg_search.groupby("Модель")["CV RMSE"].idxmin()].set_index("Модель").loc[
    ["OLS", "Ridge", "Lasso", "ElasticNet"]]
display(best_reg.style.format({"alpha": "{:.4g}", "Train RMSE": "{:.2f}", "CV RMSE": "{:.2f}", "CV std": "{:.2f}",
                               "CV SE": "{:.2f}", "G": "{:.2f}"}, na_rep="—"))
for name in ["Ridge", "Lasso"]:
    part = reg_search[reg_search["Модель"] == name]
    a = best_reg.loc[name, "alpha"]
    print(f"{name}: найкраще alpha = {a:.4g}; на межі сітки: {a in (part['alpha'].min(), part['alpha'].max())}")

Оцінено конфігурацій: 42; час: 99 с; попереджень про незбіжність: 0


,d,alpha,l1_ratio,Train RMSE,CV RMSE,CV std,CV SE,G,ConvergenceWarning
Модель,,,,,,,,,
OLS,5,—,—,64.54,260.16,203.01,90.79,195.63,0
Ridge,5,31.62,—,66.64,119.53,35.64,15.94,52.89,0
Lasso,5,0.1,—,65.93,111.36,28.49,12.74,45.43,0
ElasticNet,5,0.03162,0.800000,66.98,118.76,35.04,15.67,51.77,0


Ridge: найкраще alpha = 31.62; на межі сітки: False
Lasso: найкраще alpha = 0.1; на межі сітки: False


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharey=True)
ols = best_reg.loc["OLS"]
for ax, name in zip(axes, ["Ridge", "Lasso", "ElasticNet"]):
    part = reg_search[reg_search["Модель"] == name]
    if name == "ElasticNet":
        for r, color in zip(L1_RATIOS, ["#3f7f93", "#6a8f3f", "#8a6aa8"]):
            q = part[part["l1_ratio"] == r]
            ax.plot(q["alpha"], q["CV RMSE"], marker="o", color=color, label=f"CV, l1_ratio={r}")
            ax.plot(q["alpha"], q["Train RMSE"], ls="--", color=color, alpha=0.6)
    else:
        ax.plot(part["alpha"], part["Train RMSE"], marker="o", ls="--", color="#3f7f93", label="Train RMSE")
        ax.plot(part["alpha"], part["CV RMSE"], marker="o", color="#c46a5a", label="CV RMSE")
        best_a = best_reg.loc[name, "alpha"]
        ax.axvline(best_a, color="gray", ls=":", label=f"обране alpha = {best_a:.3g}")
    ax.axhline(ols["CV RMSE"], color="black", ls="-.", lw=1, label=f"OLS d=5 CV = {ols['CV RMSE']:.0f}")
    ax.axhline(complexity.loc[0, "CV RMSE"], color="#6a8f3f", ls=":", lw=1.5, label="OLS d=1 CV = 108.8")
    ax.set_xscale("log")
    ax.set(title=f"{name}, d = 5", xlabel="alpha (log)", ylabel="RMSE")
    ax.legend(fontsize=7)
axes[0].set_ylim(50, 300)
fig.tight_layout()
fig.savefig(FIGURES / "regularization_strength.png", dpi=160, bbox_inches="tight")
plt.show()

Для ElasticNet суцільні лінії — CV RMSE, пунктирні — train RMSE.

## 8. Аналіз ефекту регуляризації та коефіцієнтів

Для аналізу коефіцієнтів кожна конфігурація додатково навчається на всьому train (test не використовується). Норми рахуються по 251 поліноміальному коефіцієнту при стандартизованих ознаках.

In [8]:
def fit_coefficients(name, alpha=None, l1_ratio=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        model = make_pipeline(REG_DEGREE, make_estimator(name, alpha, l1_ratio)).fit(X_train, y_train)
    n_poly = comb(p + REG_DEGREE, REG_DEGREE) - 1
    return model.named_steps["model"].coef_[:n_poly]


coef_rows = []
for name in ["OLS", "Ridge", "Lasso", "ElasticNet"]:
    row = best_reg.loc[name]
    coef = fit_coefficients(name, row["alpha"], row["l1_ratio"])
    coef_rows.append({"Модель": name, "alpha": row["alpha"], "l1_ratio": row["l1_ratio"], "||w||₁": np.abs(coef).sum(),
                      "||w||₂": np.sqrt((coef ** 2).sum()), "Ненульових": int((np.abs(coef) > 1e-8).sum())})
coef_summary = pd.DataFrame(coef_rows).set_index("Модель")
display(coef_summary.style.format({"alpha": "{:.4g}", "||w||₁": "{:,.1f}", "||w||₂": "{:,.1f}"}, na_rep="—"))

path_rows = []
ridge_path, lasso_path = [], []
for a in RIDGE_ALPHAS:
    c = fit_coefficients("Ridge", a)
    ridge_path.append(c)
    path_rows.append({"Модель": "Ridge", "alpha": a, "||w||₂": np.sqrt((c ** 2).sum()), "Ненульових": int((np.abs(c) > 1e-8).sum())})
for a in L1_ALPHAS:
    c = fit_coefficients("Lasso", a)
    lasso_path.append(c)
    path_rows.append({"Модель": "Lasso", "alpha": a, "||w||₂": np.sqrt((c ** 2).sum()), "Ненульових": int((np.abs(c) > 1e-8).sum())})
paths = pd.DataFrame(path_rows)
paths.to_csv(RESULTS / "coefficient_paths.csv", index=False)
display(paths.style.format({"alpha": "{:.4g}", "||w||₂": "{:,.1f}"}))

,alpha,l1_ratio,||w||₁,||w||₂,Ненульових
Модель,,,,,
OLS,—,—,"51,831,894.6","15,602,671.8",251
Ridge,31.62,—,"1,373.8",132.0,251
Lasso,0.1,—,613.8,158.3,58
ElasticNet,0.03162,0.800000,733.6,89.2,165


,Модель,alpha,||w||₂,Ненульових
0,Ridge,0.01,"3,738.6",251
1,Ridge,0.03162,"2,246.0",251
2,Ridge,0.1,"1,337.4",251
3,Ridge,0.3162,782.2,251
4,Ridge,1,456.6,251
5,Ridge,3.162,280.5,251
6,Ridge,10,185.9,251
7,Ridge,31.62,132.0,251
8,Ridge,100,92.9,251
9,Ridge,316.2,67.8,251


In [9]:
ridge_path, lasso_path = np.array(ridge_path), np.array(lasso_path)
poly_names = PolynomialFeatures(REG_DEGREE, include_bias=False).fit(X_train[NUMERIC_FEATURES]).get_feature_names_out(NUMERIC_FEATURES)
top = np.argsort(-np.abs(lasso_path[1]))[:8]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx in range(ridge_path.shape[1]):
    axes[0].plot(RIDGE_ALPHAS, ridge_path[:, idx], color="gray", alpha=0.15, lw=0.8)
    axes[1].plot(L1_ALPHAS, lasso_path[:, idx], color="gray", alpha=0.15, lw=0.8)
for idx in top:
    axes[0].plot(RIDGE_ALPHAS, ridge_path[:, idx], lw=2, label=poly_names[idx])
    axes[1].plot(L1_ALPHAS, lasso_path[:, idx], lw=2, label=poly_names[idx])
for ax, title in [(axes[0], "Регуляризаційний шлях Ridge"), (axes[1], "Регуляризаційний шлях Lasso")]:
    ax.set_xscale("log")
    ax.set_yscale("symlog", linthresh=1)
    ax.set(title=title, xlabel="alpha (log)", ylabel="стандартизований коефіцієнт (symlog)")
axes[1].legend(fontsize=7, title="8 найбільших у Lasso", loc="upper right")
r, l = paths[paths["Модель"] == "Ridge"], paths[paths["Модель"] == "Lasso"]
axes[2].plot(r["alpha"], r["||w||₂"], marker="o", label="Ridge ||w||₂", color="#3f7f93")
axes[2].plot(l["alpha"], l["||w||₂"], marker="o", label="Lasso ||w||₂", color="#c46a5a")
axes[2].set_xscale("log")
axes[2].set_yscale("log")
axes[2].set(title="Норма і кількість ненульових коефіцієнтів", xlabel="alpha (log)", ylabel="||w||₂ (log)")
ax2 = axes[2].twinx()
ax2.plot(r["alpha"], r["Ненульових"], ls="--", color="#3f7f93", label="Ridge: ненульових")
ax2.plot(l["alpha"], l["Ненульових"], ls="--", color="#c46a5a", label="Lasso: ненульових")
ax2.set_ylabel("Ненульових коефіцієнтів (пунктир)")
ax2.grid(False)
h1, l1 = axes[2].get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
axes[2].legend(h1 + h2, l1 + l2, fontsize=7, loc="center left")
fig.tight_layout()
fig.savefig(FIGURES / "regularization_paths.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Перевірка стійкості вибору `alpha`

Процедуру вибору повторено для п'яти допустимих часових схем: `TimeSeriesSplit(n_splits = 4, 5, 6, 7, 8)`. Кожна схема має інші межі fold-ів і інший розмір validation. Для ElasticNet використовується `l1_ratio`, обраний в основному пошуку.

In [10]:
best_l1_ratio = best_reg.loc["ElasticNet", "l1_ratio"]
stability_rows = []
for n_splits in [4, 5, 6, 7, 8]:
    scheme = TimeSeriesSplit(n_splits=n_splits)
    for name, alphas in [("Ridge", RIDGE_ALPHAS), ("Lasso", L1_ALPHAS), ("ElasticNet", L1_ALPHAS)]:
        results = pd.DataFrame([evaluate(name, a, best_l1_ratio if name == "ElasticNet" else None, cv_scheme=scheme)
                                for a in alphas])
        best = results.loc[results["CV RMSE"].idxmin()]
        coef = fit_coefficients(name, best["alpha"], best["l1_ratio"])
        stability_rows.append({"Модель": name, "n_splits": n_splits, "alpha": best["alpha"], "CV RMSE": best["CV RMSE"],
                               "||w||₂": np.sqrt((coef ** 2).sum()), "Ненульових": int((np.abs(coef) > 1e-8).sum()),
                               "nonzero_set": frozenset(np.flatnonzero(np.abs(coef) > 1e-8))})
stability = pd.DataFrame(stability_rows)
stability.drop(columns="nonzero_set").to_csv(RESULTS / "stability_runs.csv", index=False)


def jaccard_mean(sets):
    sets = list(sets)
    values = [len(a & b) / len(a | b) for i, a in enumerate(sets) for b in sets[i + 1:] if a | b]
    return np.mean(values) if values else np.nan


stability_summary = stability.groupby("Модель").agg(
    alpha_min=("alpha", "min"), alpha_max=("alpha", "max"), cv_mean=("CV RMSE", "mean"), cv_sd=("CV RMSE", "std"),
    norm_min=("||w||₂", "min"), norm_max=("||w||₂", "max"), nonzero_min=("Ненульових", "min"), nonzero_max=("Ненульових", "max"),
    jaccard=("nonzero_set", jaccard_mean)).loc[["Ridge", "Lasso", "ElasticNet"]]
stability_summary.to_csv(RESULTS / "stability_summary.csv")
display(stability.drop(columns="nonzero_set").style.format({"alpha": "{:.4g}", "CV RMSE": "{:.2f}", "||w||₂": "{:,.1f}"}))
display(stability_summary.style.format({"alpha_min": "{:.4g}", "alpha_max": "{:.4g}", "cv_mean": "{:.2f}", "cv_sd": "{:.2f}",
                                        "norm_min": "{:,.1f}", "norm_max": "{:,.1f}", "jaccard": "{:.2f}"}))

,Модель,n_splits,alpha,CV RMSE,||w||₂,Ненульових
0,Ridge,4,100,138.03,92.9,251
1,Lasso,4,0.1,127.03,158.3,58
2,ElasticNet,4,0.03162,134.58,89.2,165
3,Ridge,5,31.62,119.53,132.0,251
4,Lasso,5,0.1,111.36,158.3,58
5,ElasticNet,5,0.03162,118.76,89.2,165
6,Ridge,6,31.62,115.64,132.0,251
7,Lasso,6,0.1,109.65,158.3,58
8,ElasticNet,6,0.03162,115.57,89.2,165
9,Ridge,7,100,136.12,92.9,251


,alpha_min,alpha_max,cv_mean,cv_sd,norm_min,norm_max,nonzero_min,nonzero_max,jaccard
Модель,,,,,,,,,
Ridge,10,100,123.95,12.42,92.9,185.9,251,251,1.00
Lasso,0.1,0.3162,112.96,8.40,117.4,158.3,42,58,0.81
ElasticNet,0.03162,0.1,120.46,9.43,63.4,89.2,155,165,0.87


`cv_mean`/`cv_sd` — середнє і std найкращого CV RMSE між п'ятьма схемами; вони не порівнюються напряму з основною таблицею, бо різні схеми мають різні validation-періоди. `jaccard` — середня подібність множин ненульових коефіцієнтів між схемами (1 — однакові множини).

## 10. Вибір фінальної моделі (до відкриття test)

In [11]:
candidates = []
for _, r in complexity.iterrows():
    candidates.append({"Модель": f"OLS d={int(r['d'])}", "d": int(r["d"]), "Ознак": int(r["Усього ознак"]), "Гіперпараметри": "—",
                       "Train RMSE": r["Train RMSE"], "CV RMSE": r["CV RMSE"], "CV SE": r["CV SE"], "G": r["G = CV − train"],
                       "||w||₂": np.nan, "Ненульових": comb(p + int(r["d"]), int(r["d"])) - 1, "Час CV, с": r["Час CV, с"]})
for name in ["Ridge", "Lasso", "ElasticNet"]:
    r = best_reg.loc[name]
    params = f"alpha={r['alpha']:.3g}" + (f", l1_ratio={r['l1_ratio']}" if name == "ElasticNet" else "")
    candidates.append({"Модель": f"{name} d=5", "d": 5, "Ознак": 306, "Гіперпараметри": params, "Train RMSE": r["Train RMSE"],
                       "CV RMSE": r["CV RMSE"], "CV SE": r["CV SE"], "G": r["G"], "||w||₂": coef_summary.loc[name, "||w||₂"],
                       "Ненульових": coef_summary.loc[name, "Ненульових"], "Час CV, с": np.nan})
candidates = pd.DataFrame(candidates)
candidates.loc[candidates["Модель"] == "OLS d=5", "||w||₂"] = coef_summary.loc["OLS", "||w||₂"]
best_idx = candidates["CV RMSE"].idxmin()
threshold_1se = candidates.loc[best_idx, "CV RMSE"] + candidates.loc[best_idx, "CV SE"]
candidates["≤ RMSE_min + SE"] = candidates["CV RMSE"] <= threshold_1se
candidates.to_csv(RESULTS / "final_comparison.csv", index=False)
display(candidates.style.format({"Train RMSE": "{:.2f}", "CV RMSE": "{:,.2f}", "CV SE": "{:,.2f}", "G": "{:,.2f}",
                                 "||w||₂": "{:,.1f}", "Час CV, с": "{:.1f}"}, na_rep="—"))
print(f"Мінімальний CV RMSE: {candidates.loc[best_idx, 'Модель']} = {candidates.loc[best_idx, 'CV RMSE']:.2f}; "
      f"поріг 1-SE = {threshold_1se:.2f}")
admissible = candidates[candidates["≤ RMSE_min + SE"]].sort_values(["Ознак", "CV RMSE"])
print("Допустимі за 1-SE:", list(admissible["Модель"]))
final_name = admissible.iloc[0]["Модель"]
print("Найпростіша допустима (фінальна):", final_name)

,Модель,d,Ознак,Гіперпараметри,Train RMSE,CV RMSE,CV SE,G,||w||₂,Ненульових,"Час CV, с",≤ RMSE_min + SE
0,OLS d=1,1,60,—,69.14,108.76,9.68,39.62,—,5,0.2,True
1,OLS d=2,2,75,—,67.43,108.90,10.78,41.46,—,20,0.3,True
2,OLS d=3,3,110,—,65.79,109.52,10.12,43.72,—,55,0.7,True
3,OLS d=4,4,180,—,65.15,158.97,33.54,93.81,—,125,0.8,False
4,OLS d=5,5,306,—,64.54,260.16,90.79,195.63,"15,602,671.8",251,1.2,False
5,OLS d=6,6,516,—,63.62,"2,337.20","1,053.36","2,273.58",—,461,2.0,False
6,OLS d=8,8,1341,—,61.06,"205,895.28","120,316.59","205,834.21",—,1286,8.5,False
7,Ridge d=5,5,306,alpha=31.6,66.64,119.53,15.94,52.89,132.0,251,—,False
8,Lasso d=5,5,306,alpha=0.1,65.93,111.36,12.74,45.43,158.3,58,—,True
9,ElasticNet d=5,5,306,"alpha=0.0316, l1_ratio=0.8",66.98,118.76,15.67,51.77,89.2,165,—,False


Мінімальний CV RMSE: OLS d=1 = 108.76; поріг 1-SE = 118.44
Допустимі за 1-SE: ['OLS d=1', 'OLS d=2', 'OLS d=3', 'Lasso d=5']
Найпростіша допустима (фінальна): OLS d=1


**Обґрунтування.** Обидва критерії ведуть до простої моделі: OLS d=1 має найменший середній CV RMSE (108,76) і найменшу кількість ознак. У межі `RMSE_min + SE = 118,44` потрапляють OLS d=2, OLS d=3 і Lasso d=5 (111,36), але всі вони складніші і не кращі. Ridge d=5 (119,53) та ElasticNet d=5 (118,76) — трохи за межею. Регуляризація успішно «приборкала» перенавчений простір, але не дала відтворюваної переваги над простою моделлю, тому складність не виправдана. Вимога інтерпретованості веде до того самого вибору: 60 коефіцієнтів проти 306.

## 11. Фінальне оцінювання

Обрану модель навчено на всьому train і один раз оцінено на test. Для ілюстрації наслідків перенавчання так само один раз оцінено наперед зафіксовані контрольні моделі — OLS d=5 і найкращий Ridge d=5; їхні результати не використовуються для зміни вибору.

In [12]:
final_specs = {
    f"{final_name} (обрана)": make_pipeline(1, LinearRegression()),
    "OLS d=5 (контроль)": make_pipeline(5, LinearRegression()),
    f"Ridge d=5, alpha={best_reg.loc['Ridge', 'alpha']:.3g} (контроль)": make_pipeline(5, Ridge(alpha=best_reg.loc["Ridge", "alpha"])),
}
test_rows = []
for name, model in final_specs.items():
    pred = model.fit(X_train, y_train).predict(X_test)
    test_rows.append({"Модель": name, "MAE": mean_absolute_error(y_test, pred), "RMSE": root_mean_squared_error(y_test, pred),
                      "R²": r2_score(y_test, pred)})
test_results = pd.DataFrame(test_rows)
test_results.to_csv(RESULTS / "test_results.csv", index=False)
display(test_results.style.format({"MAE": "{:,.2f}", "RMSE": "{:,.2f}", "R²": "{:.3f}"}))

,Модель,MAE,RMSE,R²
0,OLS d=1 (обрана),98.80,133.84,0.632
1,OLS d=5 (контроль),954.63,"15,188.91",-4744.654
2,"Ridge d=5, alpha=31.6 (контроль)",95.60,130.09,0.652


Висновки щодо test, аудиту ШІ та підсумкові відповіді наведено в `README.md`.